In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# replace None with an integer n to print n rows
pd.set_option('display.max_rows', None)

In [2]:
# function to get T, G, M values: (starname, dataframe)

def TGM_finder(name, dataframe):
    row = dataframe.loc[dataframe['Star'] == name]
    # return T, G, M
    return row.values[0][1], row.values[0][2], row.values[0][3] 

In [3]:
# Read the HARPS-GTO csv files into dataframes

SwoP = pd.read_csv("starswithoutplanet.csv")      # Stars WithOut planets
SwP = pd.read_csv("starswithplanet.csv")          # Stars With Planets

### <font color=steelblue> Insights on the original .csv files </font>

* **SwP has 220 entries- 147 unique host stars and 220 unique planets.**

> 1. The name of planets are always "hostname x" where x = b, c, d, etc. (refer columns 'hostname_1' and 'pl_name'; Exception- rows 140 & 168). 'Star---' column is redundant wrt 'hostname_1'.
> 2. 'TeffK' is same as 'TeffK'. But 'TeffK_x' and 'Teff_y' are different.
> 3. 'logg[cm/s2]', 'logg[cm/s2]_x' and, 'logg[cm/s2]_y' are all different.

* **SwoP has 947 unique planetless stars:**

### <font color=steelblue> In the following code, we consider:</font>
> 1. star ('hostname_1') 
> 2. temperature ('TeffK') 
> 3. metallicity [Fe/H] ('[Fe/H]---_x')
> 4. gravity ('logg[cm/s2]').

## *Derive dataframes with relevant columns from the original dataframes:*

In [4]:
# step 1: make a copy of parameter columns
cSwP = SwP[['hostname_1','TeffK', 'logg[cm/s2]', '[Fe/H]---_x']].copy()
cSwoP = SwoP[['Star---','TeffK', 'logg[cm/s2]', '[Fe/H]---_x']].copy()

# rename the 'Star---' column for convenience
cSwP = cSwP.rename(columns={'hostname_1': 'Star'}).copy()
cSwoP = cSwoP.rename(columns={'Star---': 'Star'}).copy()

# drop duplicates in nSwP: original nSwP has 220 entries but only 147 unique stars
cSwP = cSwP.drop_duplicates().copy()
cSwP = cSwP.reset_index(drop=True).copy()

## *Rank the nearest neighbours*

In [5]:
# set the cutoff radius for temperature
radius = 160

In [6]:
def ranker(star_wp):
        # copy the row from cSwP in consideration
        tdf1 = cSwP.loc[cSwP['Star'] == star_wp]

        # extract the T, G, M for the star_wp
        T, G, M = TGM_finder(star_wp, cSwP)

        # get stars in cSwoP within radius of star_wp
        tdf2 = cSwoP.loc[np.abs(cSwoP['TeffK']-T)<=radius]

        # merge tdf1 and tdf2 rowwise such that first row shows the star_wp
        tdf3 = pd.concat([tdf1, tdf2], ignore_index = True)

        # add columns to tdf3 showing difference between a star's TGM values and star_wp's TGM values

        tdf3[u'ΔT'] = (tdf3['TeffK']-T).abs()
        tdf3[u'ΔG'] = (tdf3['logg[cm/s2]']-G).abs()
        tdf3[u'ΔM'] = (tdf3['[Fe/H]---_x']-M).abs()

        # add columns to tdf3 with ranking of each parameter
        for key, col in [[u'ΔT','T_rank'], [u'ΔG','G_rank'], [u'ΔM', 'M_rank']]:
            x = np.zeros(len(tdf3[key]))
            x[tdf3[key].argsort()]=np.arange(len(tdf3[key]))
            tdf3[col] = x

        # add a mean ranking column for each star (note that star_wp will have rank 0)
        tdf3['mean_rank'] = (tdf3.T_rank+ tdf3.G_rank+tdf3.M_rank)/3.

        # extract only relevant columns from tdf3 as result1
        result1 = tdf3.sort_values('mean_rank').reset_index(drop=True)

        # sort result1 based on ranking
        result1 = result1[['Star','TeffK', 'logg[cm/s2]', '[Fe/H]---_x']]
        
        return tdf3, result1
    
# tdf3 stores calculations based on continuous parameters and result1 stores only relevant columns

# <font color=midnightblue> OUTPUT SECTION </font>
***
**This is the list of stars with planets from HARPS- GTO data**

In [7]:
cSwP

,Star,TeffK,logg[cm/s2],[Fe/H]---_x
0,BD-08 2823,4816,4.43,-0.06
1,BD-17 63,4665,4.19,0.01
2,HD 100777,5536,4.33,0.25
3,HD 10180,5911,4.39,0.08
4,HD 101930,5164,4.40,0.13
5,HD 102117,5657,4.31,0.28
6,HD 102365,5629,4.44,-0.29
7,HD 102843,5432,4.35,0.16
8,HD 103197,5250,4.45,0.22
9,HD 103720,5017,4.43,-0.02


**Enter the row number (0-146) of any star from the above list to find similar stars without planets:**

In [8]:
rollno = int(input())
full_calculation, result = ranker(cSwP['Star'][rollno])

1


**The first row is the star you chose. The rest are its neighbours printed in descending degree of similarity.**

In [9]:
result

,Star,TeffK,logg[cm/s2],[Fe/H]---_x
0,BD-17 63,4665,4.19,0.01
1,HIP21934,4674,4.19,0.03
2,HD11938,4703,4.25,0.01
3,HD45977,4689,4.30,0.03
4,BD-064196,4688,4.30,-0.03
5,BD-034797,4622,4.21,0.06
6,HIP116374,4626,4.31,0.01
7,HD108935,4724,4.30,0.02
8,HIP67126,4681,4.20,-0.19
9,HD297396,4717,4.30,0.06
